### 1. Operation 파일 파악

In [13]:
import pandas as pd 
import matplotlib.pyplot as plt 

op_data_path = '../data/Train/Train1_Operation.csv'
df_op1 = pd.read_csv(op_data_path, encoding= 'cp949')

df_op1.head(10)

,Time[sec],Torque[Nm],Motor speed[rpm],TC SP Front[℃],TC SP Rear[℃]
0,11,-4.531,498.0,10.279,10.389
1,21,-4.457,736.0,10.287,10.396
2,31,-4.508,736.0,10.315,10.414
3,41,-4.224,738.0,10.369,10.471
4,51,-4.335,738.0,10.437,10.534
5,61,-4.607,740.0,10.518,10.621
6,71,-4.504,738.0,10.601,10.712
7,81,-4.121,739.0,10.690,10.806
8,91,-3.887,739.0,10.777,10.902
9,101,-3.860,740.0,10.865,10.993


In [8]:
df_op1.tail()

,Time[sec],Torque[Nm],Motor speed[rpm],TC SP Front[℃],TC SP Rear[℃]
7518,75211,-11.633,903.0,67.575,64.333
7519,75221,-11.864,944.0,67.738,64.345
7520,75231,-14.867,981.0,68.184,64.423
7521,75241,-14.735,982.0,69.049,64.582
7522,75251,-25.602,976.0,70.273,64.830


In [9]:
df_op1['Time[sec]'].describe()

count     7523.000000
mean     37630.752625
std      21724.385940
min         11.000000
25%      18821.000000
50%      37631.000000
75%      56441.000000
max      75251.000000
Name: Time[sec], dtype: float64

In [10]:
df_op1['Time[sec]'].diff().value_counts()

Time[sec]
10.0    7502
11.0      20
Name: count, dtype: int64

### 2. Vibration 파일 확인

In [ ]:
from nptdms import TdmsFile 
import numpy as np

tdms_path = '../data/Train/Train1_Vibration/000001.tdms'
vib_000001 = TdmsFile.read(tdms_path)

print("=== Groups in TDMS ===")
for group in vib_000001.groups():
    print("Group:", group.name)

    print("\n=== Channels by Group ===")
for group in vib_000001.groups():
    print(f"[Group: {group.name}]")
    for channel in group.channels():
        print("  Channel:", channel.name)

=== Groups in TDMS ===
Group: Vibration

=== Channels by Group ===
[Group: Vibration]
  Channel: CH1
  Channel: CH2
  Channel: CH3
  Channel: CH4


In [ ]:
group_name = "Vibration"   # 실제 group 이름
channel_name = "CH1"     # 실제 channel 이름

channel = vib_000001[group_name][channel_name]
data = channel[:]   # numpy array

print("Data length:", len(data))

print("Basic stats:")
print("  mean :", np.mean(data))
print("  std  :", np.std(data))
print("  min  :", np.min(data))
print("  max  :", np.max(data))

# 첫 60초 수집한 데이터

Data length: 1536000
Sampling rate (if stored in properties): Unknown
Basic stats:
  mean : -0.018695857
  std  : 0.14936174
  min  : -0.9567261
  max  : 0.94776154


### 2-1. 진동 데이터 병합

In [ ]:
from nptdms import TdmsFile

tdms = TdmsFile.read("../data/train/Train1_Vibration/000001.tdms")
group = tdms.groups()[0]  

for ch in group.channels():
    print("Channel:", ch.name)
    print("  properties:")
    for k, v in ch.properties.items():
        print(f"    {k}: {v}")
    print()

Channel: CH1
  properties:

Channel: CH2
  properties:

Channel: CH3
  properties:

Channel: CH4
  properties:



In [ ]:
import glob
import numpy as np
import pandas as pd
from nptdms import TdmsFile

vib_dir = '../data/Train/Train1_Vibration'
vib_files = sorted(glob.glob(f"{vib_dir}/*.tdms"))

print('number of TDMS files: ', len(vib_files))

fs = 25600       # 샘플링 주파수 (Hz)
sec_len = fs     # 1초당 샘플 수
t_step = 600 # 10분 단위
win = 60 # 측정 구간 1분

rows= []

for file_idx, fpath in enumerate(vib_files) :
    # 시간 정보 : i번째 파일 기준 시각
    # i=0 > t_i = 0*600 + 60 = 60초
    file_start_sec = file_idx * t_step

    tdms = TdmsFile.read(fpath) 
    group = tdms.groups()[0] # Vibration

    ch1 = tdms[group.name]["CH1"][:]
    ch2 = tdms[group.name]["CH2"][:]
    ch3 = tdms[group.name]["CH3"][:]
    ch4 = tdms[group.name]["CH4"][:]

    # 파일 내 데이터가 몇 초까지 있는지(60초/그 미만)
    n_samples = len(ch1)
    

    # 하나의 row로 정리
    row = {
        "bearing_id": 1,          # Train1
        "sample_idx": i+1,
        "t_sec": t_i,
    }

    # 채널별로 prefix 붙여서 넣기
    for k, v in f_ch1.items():
        row[f"CH1_{k}"] = v
    for k, v in f_ch2.items():
        row[f"CH2_{k}"] = v
    for k, v in f_ch3.items():
        row[f"CH3_{k}"] = v
    for k, v in f_ch4.items():
        row[f"CH4_{k}"] = v

    rows.append(row)

df_vib_train1 = pd.DataFrame(rows)

df_vib_train1.head()

number of TDMS files:  126


,bearing_id,sample_idx,t_sec,CH1_rms,CH1_peak,CH1_mean,CH1_std,CH2_rms,CH2_peak,CH2_mean,CH2_std,CH3_rms,CH3_peak,CH3_mean,CH3_std,CH4_rms,CH4_peak,CH4_mean,CH4_std
0,1,1,60,0.150527,0.956726,-0.018696,0.149362,0.196044,1.142979,0.010669,0.195754,0.125165,0.907898,0.002423,0.125141,0.170665,1.020622,-0.000023,0.170665
1,1,2,660,0.150666,0.977039,-0.019668,0.149376,0.198770,1.196289,0.010332,0.198501,0.126157,0.831699,0.001667,0.126146,0.172179,1.043129,0.000311,0.172178
2,1,3,1260,0.148393,1.010036,-0.018984,0.147173,0.201856,1.156330,0.009690,0.201623,0.125212,0.855923,0.001739,0.125200,0.171102,1.188374,0.000009,0.171102
3,1,4,1860,0.150413,0.965881,-0.019041,0.149203,0.368733,1.986122,0.010974,0.368570,0.147569,1.207066,0.002415,0.147549,0.250142,1.948357,0.001089,0.250140
4,1,5,2460,0.148550,0.852966,-0.018379,0.147409,0.420878,2.383423,0.010949,0.420736,0.156140,1.243114,0.002801,0.156115,0.268654,2.162933,0.000768,0.268653


In [21]:
df_vib_train1.tail()

,bearing_id,sample_idx,t_sec,CH1_rms,CH1_peak,CH1_mean,CH1_std,CH2_rms,CH2_peak,CH2_mean,CH2_std,CH3_rms,CH3_peak,CH3_mean,CH3_std,CH4_rms,CH4_peak,CH4_mean,CH4_std
121,1,122,72660,0.319265,3.545761,-0.016123,0.318858,0.940738,5.155945,0.009201,0.940693,0.285229,1.783562,0.004615,0.285191,0.565167,4.757977,0.001183,0.565166
122,1,123,73260,0.306077,4.213810,-0.016654,0.305624,0.910779,6.121731,0.008528,0.910739,0.283966,1.763439,0.003795,0.283941,0.554638,5.277061,0.001067,0.554637
123,1,124,73860,0.298595,4.388714,-0.016445,0.298142,0.863366,5.196476,0.008158,0.863327,0.275342,1.971722,0.003965,0.275313,0.528560,5.274200,0.001303,0.528559
124,1,125,74460,0.296784,3.857517,-0.015736,0.296367,0.843418,6.298542,0.008511,0.843375,0.266845,1.544094,0.003795,0.266818,0.519220,3.981590,0.001530,0.519218
125,1,126,75060,0.432889,11.025524,-0.015912,0.432597,2.065830,11.177444,0.008910,2.065811,0.478862,2.913952,0.004305,0.478842,1.211021,7.343197,0.002052,1.211020


In [20]:
df_vib_train1.info()

<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   bearing_id  126 non-null    int64  
 1   sample_idx  126 non-null    int64  
 2   t_sec       126 non-null    int64  
 3   CH1_rms     126 non-null    float32
 4   CH1_peak    126 non-null    float32
 5   CH1_mean    126 non-null    float32
 6   CH1_std     126 non-null    float32
 7   CH2_rms     126 non-null    float32
 8   CH2_peak    126 non-null    float32
 9   CH2_mean    126 non-null    float32
 10  CH2_std     126 non-null    float32
 11  CH3_rms     126 non-null    float32
 12  CH3_peak    126 non-null    float32
 13  CH3_mean    126 non-null    float32
 14  CH3_std     126 non-null    float32
 15  CH4_rms     126 non-null    float32
 16  CH4_peak    126 non-null    float32
 17  CH4_mean    126 non-null    float32
 18  CH4_std     126 non-null    float32
dtypes: float32(16), int64(3)
memory usage: 1

### 3. 고장 시점(구간) 정의

In [14]:
df_op1.info()

<class 'pandas.DataFrame'>
RangeIndex: 7523 entries, 0 to 7522
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Time[sec]           7523 non-null   int64  
 1     Torque[Nm]        7523 non-null   float64
 2     Motor speed[rpm]  7523 non-null   float64
 3     TC SP Front[℃]    7523 non-null   float64
 4     TC SP Rear[℃]     7523 non-null   float64
dtypes: float64(4), int64(1)
memory usage: 294.0 KB


In [ ]:
# Train1
cond_temp = (df_op1['  TC SP Front[℃]'] >= 200) | (df_op1['  TC SP Rear[℃]'] >= 200)
cond_torque = (df_op1['  Torque[Nm]'] <= -20)
cond_fail = cond_temp | cond_torque

# 실제 고장 시간 / 고장 시간 대체(구간 중 제일 빠른 시간)
last_time = df_op1['Time[sec]'].iloc[-1]

if cond_fail.any() :
    idx_fail = cond_fail.idxmax()
    t_fail_op1 = df_op1.loc[idx_fail, 'Time[sec]']
else :
    t_fail_op1 = last_time + 1.0




np.int64(75251)